# 2. Feature Engineering & Exploration

Risk Factors: https://www.cdc.gov/heart-disease/risk-factors/index.html   

Several health conditions, your lifestyle, and your age and family history can increase your risk for heart disease. These are called risk factors. Key risk factors for heart disease include:

- High blood pressure
- High cholesterol
- Smoking

Some risk factors for heart disease cannot be controlled, such as your age or family history. But you can take steps to lower your risk by changing the factors you can control.
In addition

- Polynomial features (e.g., age², bmi²) --> To weigh these heavier
- Pulse Pressure = systolic_bp−diastolic_bp
- bmi * cholesterol: could capture the compounded risk of high weight and high lipids --> High BMI (overweight/obesity) is strongly associated with adverse lipid profiles, specifically increasing LDL ("bad") cholesterol and decreasing HDL ("good") cholesterol


If you use a linear model, these highly correlated features can cause instability. You might consider:
Mean Arterial Pressure (MAP): A weighted average of your two BP readings:
Mean Arterial Pressure: $MAP=diastolic + \frac{1}{3} (systolic- diastolic)$

In [1]:
# Load Data
import pandas as pd

df = pd.read_csv("data/raw/cardiovascular_risk_dataset.csv")
df.head()

,Patient_ID,age,bmi,systolic_bp,diastolic_bp,cholesterol_mg_dl,resting_heart_rate,smoking_status,daily_steps,stress_level,physical_activity_hours_per_week,sleep_hours,family_history_heart_disease,diet_quality_score,alcohol_units_per_week,heart_disease_risk_score,risk_category
0,1,62,25.0,142,93,247,72,Never,11565,3,5.6,8.2,No,7,0.7,28.1,Medium
1,2,54,29.7,158,101,254,74,Current,4036,8,0.5,6.7,No,5,4.5,63.0,High
2,3,46,36.2,170,113,276,80,Current,3043,9,0.4,4.0,No,1,20.8,73.1,High
3,4,48,30.4,153,98,230,73,Former,5604,5,0.6,8.0,No,4,8.5,39.5,Medium
4,5,46,25.3,139,87,206,69,Current,7464,1,2.0,6.1,No,5,3.6,29.3,Medium


## 2.1 Data Cleaning

Before exploring which additional features could be created to consider the risk factors, we clean our original dataframe, based on what we found out in the EDA. As we did not find any missing values or duplicates, we will only drop columns. We will drop Patient_ID, as it is a unique identifier without any predictive value, and we will drop heart_disease_risk_score, as it directly encodes with our target (risk_category), which would cause leakage. 

In [2]:
# Drop Patient_ID and heart_disease_risk_score in a new dataframe so we do not change our original one 
df_clean = df.copy()

df_clean = df_clean.drop(columns=['Patient_ID', 'heart_disease_risk_score'])

print("Cleaned dataframe shape:", df_clean.shape)
print("Columns: ", df_clean.columns.tolist())

Cleaned dataframe shape: (5500, 15)
Columns:  ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol_mg_dl', 'resting_heart_rate', 'smoking_status', 'daily_steps', 'stress_level', 'physical_activity_hours_per_week', 'sleep_hours', 'family_history_heart_disease', 'diet_quality_score', 'alcohol_units_per_week', 'risk_category']


## 2.2 Train-Test Split
We perform the train–test split before feature engineering because any decision about creating, selecting, or removing features must be based solely on the training data; otherwise, information from the test set would indirectly influence the model, leading to data leakage and overly optimistic performance estimates.

In [3]:
from sklearn.model_selection import train_test_split

# Separating features from target variable
X = df_clean.drop(columns=['risk_category'])
y = df_clean['risk_category']

# Split into training and tetsing set 80/20 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y , shuffle = True) # check if we should really stratify -> justify it in text 

print("Training set size:", X_train.shape)
print("Test set size:", X_test.shape)
print("\nClass distribution in training set:\n", y_train.value_counts(normalize=True))
print("\nClass distribution in test set:\n", y_test.value_counts(normalize=True))

Training set size: (4400, 14)
Test set size: (1100, 14)

Class distribution in training set:
 risk_category
Medium    0.407955
Low       0.334318
High      0.257727
Name: proportion, dtype: float64

Class distribution in test set:
 risk_category
Medium    0.408182
Low       0.333636
High      0.258182
Name: proportion, dtype: float64


## 2.3 Feature Creation

As mentioned above, there are certain risk factors for getting a heart disease. While our dataset already provides some of these factors, by creating new features from the existing ones, we aim to better capture relationships that raw columns cannot express directly. We organise our engineered features into three categories: 
polynomial transformations, interaction features, and binned flag features.

All features are created on X_train data only to prevent leakage from the test set.

In [4]:
# using a copy of X_train dataframe to later being able to distinguish from original features and created ones 
df_engineered = X_train.copy()
df_engineered.head()

,age,bmi,systolic_bp,diastolic_bp,cholesterol_mg_dl,resting_heart_rate,smoking_status,daily_steps,stress_level,physical_activity_hours_per_week,sleep_hours,family_history_heart_disease,diet_quality_score,alcohol_units_per_week
2306,19,26.8,139,97,219,76,Current,7196,5,4.0,7.0,Yes,4,2.3
2004,36,23.2,130,84,197,79,Never,7336,5,1.4,8.6,Yes,7,5.1
4118,18,23.6,130,85,191,71,Never,8687,5,7.7,7.1,No,7,2.6
2726,37,18.7,132,85,184,57,Never,9855,4,9.1,6.8,No,9,0.1
3816,25,26.3,151,96,210,73,Current,4475,6,1.0,7.3,Yes,5,2.0


### 2.3.1 Polynominal Transformations 
By squaring a variable, we allow the model to reflect the accelerating nature of certain risk factors. For example, cardiovascular risk does not increase at a constant rate with age or BMI, but accelerates as values grow larger. 

In [5]:
# Transform variables
df_engineered['age_squared'] = df_engineered['age'] ** 2
df_engineered['bmi_squered'] = df_engineered['bmi'] ** 2 

### 2.3.2 Interaction Features
These features show how two variables together can create a stronger risk signal than each variable on its own. Interaction terms are especially helpful for logistic regression, since it cannot capture these combined effects automatically. We also combine systolic and diastolic blood pressure into summary measures like Pulse Pressure and Mean Arterial Pressure (MAP).

- Pulse Presssure: reflects the difference between systolic and diastolic blood pressure and can indicate how stiff or elastic the arteries are. Higher values are often associated with increased cardiovascular risk, making it a useful indicator of heart and vascular health.
- Mean Arterial Pressure (MAP): Mean Arterial Pressure (MAP) shows the average pressure in the arteries over one heartbeat. It is important because it reflects how much overall pressure the heart has to work against to pump blood through the body.
- BMI x Age: reflects that the cardiovascular risk linked to excess weight can increase as people get older, meaning that high BMI may have a stronger impact at higher ages.
- BMI x Cholesterol: captures that the health risk of high body weight may become stronger when cholesterol levels are also elevated, since both factors together can increase cardiovascular risk more than each one alone.
- Cholesterol/Age: this ratio helps to show how high someone’s cholesterol level is relative to their age, since elevated cholesterol at a younger age may indicate a higher long-term cardiovascular risk.

In [6]:
# Add interactions
df_engineered['pulse_pressure'] = df_engineered['systolic_bp'] - df_engineered['diastolic_bp']
df_engineered['map'] = df_engineered['diastolic_bp'] + (df_engineered['systolic_bp'] - df_engineered['diastolic_bp']) / 3
df_engineered['bmi_x_age'] = df_engineered['bmi'] * df_engineered['age']
df_engineered['bmi_x_cholesterol'] = df_engineered['bmi'] * df_engineered['cholesterol_mg_dl']
df_engineered['cholosterol_age_ratio'] = df_engineered['cholesterol_mg_dl'] / df_engineered['age']

### 2.3.3 Binned Flag Features
These features convert clinically established thresholds into binary indicators. We explicitly include boundaries supported by medical guidelines to help the model detecting meaninful thresholds more easily. This is particularly useful for tree-based models, as it provides clear and interpretable split points. The thresholds are based on WHO and established cardiology guidelines.

-> add link with WHO thresholds!! 

- Hypertension Flag: identifies individuals who meet the clinical definition of high blood pressure (systolic ≥ 140 or diastolic ≥ 90) and is important because hypertension is a major risk factor for cardiovascular disease.
- Obesity Flag: identifies individuals with a BMI ≥ 30, based on the WHO definition of obesity, and is important because obesity is strongly associated with increased cardiovascular and metabolic risk.
- High Cholesterol Flag: identifies individuals with cholesterol levels ≥ 240 mg/dl, a clinically recognized high-risk threshold, and is important because elevated cholesterol significantly increases the risk of cardiovascular disease.
- Inactivity Flag: identifies individuals taking fewer than 5,000 steps per day, a commonly used threshold for inactive behavior, and is important because low physical activity is associated with higher cardiovascular risk; our distribution analysis also showed a noticeable concentration of patients below this level (see EDA).
- Sleep Deviation: captures how much a person’s sleep duration differs from the recommended 7 hours, and is important because both too little and too much sleep have been linked to increased cardiovascular risk.
- Alcohol Deviation: measures how much a person’s weekly alcohol consumption differs from the WHO recommendation of 14 units, where negative values indicate consumption within the recommended limit and positive values reflect the amount of excess intake, which is associated with increased health risk.

In [7]:
# Add Binary/ binned features 
df_engineered['hypertension_flag'] = ((df_engineered['systolic_bp'] >= 140) | (df_engineered['diastolic_bp'] >= 90)).astype(int)
df_engineered['obesity_flag'] = (df_engineered['bmi'] >= 30).astype(int)
df_engineered['high_cholesterol_flag'] = (df_engineered['cholesterol_mg_dl'] >= 240).astype(int)
df_engineered['inactivity_flag'] = (df_engineered['daily_steps'] < 5000).astype(int)
df_engineered['sleep_deviation'] = (df_engineered['sleep_hours'] - 7).abs()
df_engineered['alcohol_deviation'] = df_engineered['alcohol_units_per_week'] - 14

In [ ]:
# bin bmi into categories (<25 (Normal), <30 (Overweight), 30+ (Obese))
# df['bmi_binned'] = pd.cut(df['bmi'], bins=[0, 25, 30, float('inf')], labels=['Normal', 'Overweight', 'Obese'])


We have engineered ... features that could potentially help our model to predict the risk category better, but also enhances interpretability for medical personel when using our final product. We will in the next step have to verify, which features will have an importance for modelling. 

In [9]:
def dataoverview(df_engineered, message = "Data Overview"):
    print(f"{message}\n")
    print("\nNumber of rows: ", df_engineered.shape[0])
    print("\nNumber of columns: ", df_engineered.shape[1])
    print("\nColumns: ", df_engineered.columns.tolist())
    print("\nData Types: \n", df_engineered.dtypes)

dataoverview(df_engineered)

Data Overview


Number of rows:  4400

Number of columns:  27

Columns:  ['age', 'bmi', 'systolic_bp', 'diastolic_bp', 'cholesterol_mg_dl', 'resting_heart_rate', 'smoking_status', 'daily_steps', 'stress_level', 'physical_activity_hours_per_week', 'sleep_hours', 'family_history_heart_disease', 'diet_quality_score', 'alcohol_units_per_week', 'age_squared', 'bmi_squered', 'pulse_pressure', 'map', 'bmi_x_age', 'bmi_x_cholesterol', 'cholosterol_age_ratio', 'hypertension_flag', 'obesity_flag', 'high_cholesterol_flag', 'inactivity_flag', 'sleep_deviation', 'alcohol_deviation']

Data Types: 
 age                                   int64
bmi                                 float64
systolic_bp                           int64
diastolic_bp                          int64
cholesterol_mg_dl                     int64
resting_heart_rate                    int64
smoking_status                       object
daily_steps                           int64
stress_level                          int64
physical_act

## 2.4 Feature Validation
 As we engineerd many features that also show redundancy and might make sense in a medical context but not for our prediction model, we will have to validate, which features to include and which ones not. To validate the features we use mutual information, which shows whih features most likely will have an impact on our target value. Lastly, we will again create a correlation matrix to identify which features which introduce redundancy and could therefore duplicate effetcs in our model. 

## 2.5 Feature Selection
In this final step before modeling, we create functions with the features that we want to include for our Logistic regression model, as well as a seperate function for features that will be used in our tree models. Our goal is to use specific features depending on the models strengths to optimize our prediction. 